In [0]:
storage_account = "ahmedretaildatalake123"
container = "salesdata"

data_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/bronze"

In [0]:
spark.conf.set(
"fs.azure.account.key.ahmedretaildatalake123.dfs.core.windows.net",
"27+qcs4hJm6Oqx7I+FqIKDEca4UyxAadR3fL1OiMC/T/h0UG0EMDD0p6upDsdbgWatq3v6+D/A+1+AStBMRF/w=="
)

In [0]:
data_path = "abfss://salesdata@ahmedretaildatalake123.dfs.core.windows.net/bronze/Sample - Superstore.csv"

df = spark.read.format("csv") \
.option("header","true") \
.option("inferSchema","true") \
.load(data_path)

df.show()

+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer Name|    Segment|      Country|           City|         State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+------------------+-----------+-------------+---------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156|2016-11-08|2016-11-11|  Second Class|   CG-12520|       Claire Gute|   Consumer|United States|      Henderson|      Kentucky|      42420|  South|FUR-BO-10001798|   

**Silver Layer**

In [0]:
df_clean = df.dropna()

In [0]:
df_clean = df_clean.dropDuplicates()

In [0]:
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Ship Date: date (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)



In [0]:
from pyspark.sql.functions import col

df_clean = df_clean.withColumn("Sales", col("Sales").cast("double")) \
                   .withColumn("Quantity", col("Quantity").cast("int")) \
                   .withColumn("Discount", col("Discount").cast("double"))

In [0]:
from pyspark.sql.functions import col

df_clean = df_clean.withColumn(
    "Revenue",
    col("Sales") - col("Discount")
)

In [0]:
from pyspark.sql.functions import expr

df_clean = df.withColumn("Sales", expr("try_cast(Sales as double)")) \
             .withColumn("Quantity", expr("try_cast(Quantity as int)")) \
             .withColumn("Discount", expr("try_cast(Discount as double)"))

In [0]:
df_clean = df_clean.dropna(subset=["Sales"])

In [0]:
df_clean.write.mode("overwrite") \
.parquet("abfss://salesdata@ahmedretaildatalake123.dfs.core.windows.net/silver/")

**Gold Layer**

In [0]:
df_gold = df_clean.groupBy("Region") \
.sum("Revenue")
df_gold.write.mode("overwrite") \
.parquet("abfss://salesdata@ahmedretaildatalake123.dfs.core.windows.net/gold/")